# Prompting, Structured Outputs, and Tool Use with Claude

In this lab, we are going to work with **one contract PDF from start to finish**.

First, we’ll simply **extract the text from the PDF**.

Then, we’ll send that text to **Claude** using a very basic prompt.

We’ll intentionally see where that prompt falls short, diagnose the problem, and improve it.

After that, we’ll see why **good prompting alone is not always enough for an application**. We’ll progressively introduce:

- Structured outputs
- Tool use
- Strict tool schemas
- Tool error handling

So, we are not building separate examples. We are improving the **same contract-processing workflow step by step**.

# Install the Libraries

First, we need two Python libraries.
> `anthropic` lets us talk to Claude through the API.
>
> `pymupdf` lets us read the text inside our PDF.

In [1]:
!pip install -q -U anthropic pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 31.4 MB/s eta 0:00:00


# Create the Anthropic Client


Now we create an Anthropic client.

Think of this client as the connection between our Python notebook and Claude.

#### ⚠️ **Warning:** Before running the code, make sure you add your own Anthropic API key where indicated.

> Keep your API key private. Do not share it publicly, upload it to GitHub, or include it in screenshots or shared notebooks.


In [ ]:
from anthropic import Anthropic

client = Anthropic(
    api_key= "Place Your Anthropic API Key Here"
)

MODEL = "claude-haiku-4-5-20251001"

# Test the Connection

Before doing anything with the contract, let's first check whether our Claude connection is working.

We’ll send a very simple test message to Claude and confirm that we get a response back.


In [3]:
response = client.messages.create(
    model=MODEL,
    max_tokens=100,
    messages=[
        {
            "role": "user",
            "content": "Say: Claude connection successful."
        }
    ]
)

print(response.content[0].text)

Claude connection successful.


# Upload the Contract PDF

Now let's upload the contract PDF that we want to process.

We’ll use this same PDF throughout the lab, so every improvement we make will build on the same contract-processing workflow.

**⚠️ Note:** **In the cell below, you need to upload a file named `AWS1.pdf`.**  
**You can download the file from the link below.**
[📥 Download AWS1.pdf](https://drive.google.com/file/d/1XSe2pSsGN1ssAbif92rvb80AnHb_Ni0F/view?usp=sharing)/Lab-2.1(Generating-Response-without-RAG)/AWS1.pdf)


In [4]:
from google.colab import files
uploaded = files.upload()

Saving AWS1.pdf to AWS1.pdf


# Extract Text Using PyMuPDF


Claude cannot magically access a local file sitting inside our notebook.

We first need to turn the PDF into something we can send to the model.

> For this example, we're going to use `PyMuPDF` to extract the text from the contract.

Once the text is extracted, we can pass it to Claude as part of our prompt.


In [7]:
def extract_text(pdf_path):
    doc = fitz.open(pdf_path)

    pages = []

    for page_number, page in enumerate(doc, start=1):
        text = page.get_text()

        pages.append({
            "page": page_number,
            "text": text
        })

    return pages

In [9]:
document = extract_text("/content/AWS1.pdf")

# First Attempt — A Dumb Prompt

Now imagine I know nothing about prompt engineering.
I have my contract text, and I simply ask Claude:

> ```text
> Tell me the important information from this contract.
> ```

Let's try exactly that and see what happens.


In [10]:
response = client.messages.create(
    model=MODEL,
    max_tokens=800,
    messages=[
        {
            "role": "user",
            "content": f"""
Tell me the important information from this contract.

{document}
"""
        }
    ]
)

print(response.content[0].text)

# AWS Customer Agreement - Key Information

## Contract Parties
- **Customer:** XYZ Software solutions
- **Service Provider:** Amazon Web Services (AWS)
- **Effective Date:** April 1, 2023
- **Contract Duration:** 12 months (ends March 31, 2024) - Does NOT auto-renew

## Financial Terms
- **Total Contract Value:** USD 35,000 annually (paid before start of work)
- **Billing:** Monthly calculation; AWS can bill more frequently if account appears fraudulent
- **Late Payment Interest:** 1.5% per month (or highest legal rate if lower)
- **Price Changes:** AWS can increase fees with 30 days' notice
- **Taxes:** Customer responsible for applicable taxes unless exemption certificates provided

## Key Responsibilities

**AWS Responsibilities:**
- Implement reasonable security measures to protect customer content
- Provide 12 months' notice before discontinuing material service functionality
- Maintain Service Level Agreements (90 days' notice for adverse changes)

**Customer Responsibilities:**

The response will probably look useful.

Claude might tell us about:

* Payments
* Termination
* Security
* Responsibilities
* 📌 Other important clauses

So, did Claude fail?

**Not really.**

The problem is actually our prompt. 🎯

Imagine our application needs exactly these fields:

```text
customer_name
effective_date
end_date
contract_value
payment_terms
auto_renewal
termination_notice
```

But our prompt never asked for those specific fields.

We simply said:

```text
Tell me the important information from this contract.
```

### 🤔 But what does **important** mean?

Important to a lawyer?

Important to finance?

Important to procurement?

Important to our application?

The model has to guess.

> 💡 **Key lesson:** If we want specific information, we need to ask for specific information.


This is our **prompt-crafting failure mode**.

### ⚠️ Failure Mode

```text
Tell me the important information.
```

Claude has to make too many decisions on its own.

### 🔍 Diagnose

We did not specify:

* What information should be extracted?
* What fields are required?
* What should happen if something is missing?
* Should Claude summarize or extract?
* What format should the result use?

So instead of blaming the model, we diagnose what was missing from our instruction.


# Resolution — Improve the Prompt ✅

Now let’s make Claude’s job **explicit**.

Instead of asking for something vague like:

```text
Tell me the important information.
```

we’ll clearly tell Claude:

* What information to extract
* Which fields are required
* What to do when a value is missing
* Whether to summarize or extract
* What format the response should use

> 💡 The more clearly we define the task, the less Claude has to guess.


In [11]:
system_prompt = """
You are a contract information extraction assistant.

Extract the following fields from the contract:

- customer_name
- effective_date
- end_date
- contract_value
- payment_terms
- auto_renewal
- termination_notice

Use only information explicitly stated in the contract.

If a value cannot be found, return NOT_FOUND.

Do not guess.
"""

And now we send the same exact PDF text again.

In [12]:
response = client.messages.create(
    model=MODEL,
    max_tokens=800,
    system=system_prompt,
    messages=[
        {
            "role": "user",
            "content": f"""
<contract>
{document}
</contract>

Extract the requested contract information.
"""
        }
    ]
)

print(response.content[0].text)

Based on my review of the AWS Customer Agreement, here are the extracted fields:

- **customer_name**: XYZ Software solutions

- **effective_date**: 1st April 2023

- **end_date**: 31st March 2024

- **contract_value**: USD 35000/- (paid annually)

- **payment_terms**: Monthly billing for fees and charges; payment due without setoff or counterclaim

- **auto_renewal**: No (The contract explicitly states "This contract does not renews automatically after it reaches the end date of contract.")

- **termination_notice**: 30 days' advance notice required for termination for convenience by AWS; customer may terminate for any reason by closing their account; 30 days to cure material breach before termination for cause


Now the output should be much closer to what our application needs.

Expected values include:

```text
customer_name: XYZ Software Solutions
effective_date: 1 April 2023
end_date: 31 March 2024
contract_value: USD 35000
payment_terms: Paid annually before the start of work
auto_renewal: No
termination_notice: 30 days
```

### What Changed?

```text
FAILURE
Vague prompt

   ↓

DIAGNOSE
"Important" was unclear

   ↓

RESOLUTION
Specify exactly what to extract
```

> 💡 **Prompt-crafting loop:** Diagnose → Improve → Try again.


We can improve the prompt further by **showing Claude examples** of what a good answer looks like.

This is called **few-shot prompting**.

> **Idea:** Don’t just tell the model what to do — show it the expected pattern.


In [13]:
system_prompt = """
You are a contract information extraction assistant.

Extract:

- customer_name
- effective_date
- end_date
- contract_value
- payment_terms
- auto_renewal
- termination_notice

Use only information explicitly stated in the contract.

If a value cannot be found, return NOT_FOUND.

Do not guess.

<example>
<sample_input>
The agreement begins on January 1, 2026.
It expires on December 31, 2026.
</sample_input>

<ideal_output>
effective_date: January 1, 2026
end_date: December 31, 2026
</ideal_output>
</example>

<example>
<sample_input>
The total contract value is USD 10,000.
</sample_input>

<ideal_output>
contract_value: USD 10,000
</ideal_output>
</example>
"""

Now run the same contract again.

In [14]:
response = client.messages.create(
    model=MODEL,
    max_tokens=800,
    system=system_prompt,
    messages=[
        {
            "role": "user",
            "content": f"""
<contract>
{document}
</contract>

Extract the requested contract information.
"""
        }
    ]
)

print(response.content[0].text)

# Contract Information Extraction

**customer_name:** XYZ Software solutions

**effective_date:** 1st April 2023

**end_date:** 31st March 2024

**contract_value:** USD 35,000 (paid annually before the start of work)

**payment_terms:** Monthly calculation and billing of fees and charges. For India-based customers: invoiced in INR (converted from USD). Payment due without setoff or counterclaim. Interest on late payments at 1.5% per month (or highest rate permitted by law, if less).

**auto_renewal:** No auto renewal (Contract does not renew automatically after it reaches the end date of contract)

**termination_notice:** 30 days' advance notice for termination for convenience by AWS. For termination for cause: 30 days from receipt of notice to cure material breach before termination takes effect.


### System Prompt, Examples & XML

Three things are now helping Claude:

```text
System prompt
→ defines Claude's job

Examples
→ show what good output looks like

XML tags
→ separate instructions, examples, and contract text
```

This usually makes the output more **consistent**.

⚠️ But we still have another problem.


# Prompting Still Does Not Guarantee Structure 🧱

Claude might return something like:

```text
Customer Name: XYZ Software Solutions
Contract Value: USD 35,000
Effective Date: 1 April 2023
```

That looks fine to a human.

But our Python application may need:

```python
print(result["contract_value"])
```

So we need a **predictable structure**.

Simply writing:

```text
Please return JSON.
```

is still just an instruction.




Now let’s define exactly what fields our Python application accepts.



In [15]:
contract_schema = {
    "type": "object",
    "properties": {
        "customer_name": {
            "type": ["string", "null"]
        },
        "effective_date": {
            "type": ["string", "null"]
        },
        "end_date": {
            "type": ["string", "null"]
        },
        "contract_value": {
            "type": ["string", "null"]
        },
        "payment_terms": {
            "type": ["string", "null"]
        },
        "auto_renewal": {
            "type": ["boolean", "null"]
        },
        "termination_notice": {
            "type": ["string", "null"]
        }
    },
    "required": [
        "customer_name",
        "effective_date",
        "end_date",
        "contract_value",
        "payment_terms",
        "auto_renewal",
        "termination_notice"
    ],
    "additionalProperties": False
}

### Ask Claude for Structured Output

Now we’ll ask Claude to return the contract details using our predefined structure.

In [19]:
response = client.messages.create(
    model=MODEL,
    max_tokens=800,
    system="""
You extract contract information.

Use only information explicitly stated in the contract.
Do not guess.
""",
    messages=[
        {
            "role": "user",
            "content": f"""
<contract>
{document}
</contract>
"""
        }
    ],
    output_config={
        "format": {
            "type": "json_schema",
            "schema": contract_schema
        }
    }
)

print(response.content[0].text)

{"customer_name": "XYZ Software solutions", "effective_date": "1st April 2023", "end_date": "31st march 2024", "contract_value": "USD 35000/-", "payment_terms": "paid annually before the start of work", "auto_renewal": false, "termination_notice": "30 days"}


Now instead of hoping Claude follows our requested structure, we're defining the structure that should be returned.

## Convert the JSON to Python

In [20]:
import json

contract_data = json.loads(response.content[0].text)

print(contract_data)

{'customer_name': 'XYZ Software solutions', 'effective_date': '1st April 2023', 'end_date': '31st march 2024', 'contract_value': 'USD 35000/-', 'payment_terms': 'paid annually before the start of work', 'auto_renewal': False, 'termination_notice': '30 days'}


And now we can access individual values directly.

In [21]:
print(contract_data["customer_name"])
print(contract_data["effective_date"])
print(contract_data["contract_value"])

XYZ Software solutions
1st April 2023
USD 35000/-


Here’s the key distinction:

```text
PROMPT
→ What should Claude do?

STRUCTURED OUTPUT
→ What shape must Claude's answer have?
```

A structured response gives us reliable data.

But it is still only **information**.

> ⚙️ What if we want our application to actually **do something** with that information?


# Tool Use

So far, Claude has only **read and extracted information** from the contract.

But in real applications, we often want Claude to do something with that information — for example:

* save contract details to a database
* search for an existing contract
* update a record
* call an API
* trigger another system or workflow

This is where **tool use** comes in.

Tool use allows Claude to connect its reasoning with **external functions and systems**. Instead of only returning text, Claude can decide when a specific function should be called and provide the required inputs.

In our contract system, we want Claude to extract the contract details and then pass those details to a tool that can **save them for later use**.

For this lab, we will keep things simple and simulate our database using a Python list.

So our flow becomes:

```text
Contract
   ↓
Claude understands the contract
   ↓
Claude decides to use the save tool
   ↓
Tool receives the extracted details
   ↓
Contract data is stored
```

We’ll use a simple Python list to act as our temporary database. The save_contract_details() function will later be called when Claude requests the tool to save the extracted contract information.

In [22]:
saved_contracts = []

In [23]:
def save_contract_details(
    customer_name,
    effective_date,
    end_date,
    contract_value
):
    record = {
        "customer_name": customer_name,
        "effective_date": effective_date,
        "end_date": end_date,
        "contract_value": contract_value
    }

    saved_contracts.append(record)

    return {
        "status": "success",
        "record": record
    }

Claude cannot directly execute our Python function.

We have to tell Claude that this capability exists.

That is what a **tool definition** does.

```text
Python function
→ performs the actual action

Tool definition
→ tells Claude the action exists
→ defines the inputs it requires
```

> The tool definition describes the capability. Our Python code performs it.


Define the Tool

In [24]:
tools = [
    {
        "name": "save_contract_details",
        "description": "Save key information extracted from a contract.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_name": {
                    "type": "string"
                },
                "effective_date": {
                    "type": "string"
                },
                "end_date": {
                    "type": "string"
                },
                "contract_value": {
                    "type": "string"
                }
            },
            "required": [
                "customer_name",
                "effective_date",
                "end_date",
                "contract_value"
            ],
            "additionalProperties": False
        }
    }
]

### Give Claude the Tool

Now we give Claude the contract and make the saving tool available.

After extracting the contract details, Claude can choose to call that tool.


In [27]:
response = client.messages.create(
    model=MODEL,
    max_tokens=800,
    system="""
Extract key contract information from the provided contract.

Use only information explicitly present in the contract.

After extracting the information, save it using the available tool.
""",
    messages=[
        {
            "role": "user",
            "content": f"""
<contract>
{document}
</contract>
"""
        }
    ],
    tools=tools
)

Let's inspect what happened.

In [28]:
print(response.stop_reason)

tool_use


In [29]:
for block in response.content:
    print(block)

TextBlock(citations=None, text="I'll extract the key contract information from this AWS Customer Agreement.", type='text')
ToolUseBlock(id='toolu_01UAR82gF3o7QZMfQkYEjoBa', caller=DirectCaller(type='direct'), input={'customer_name': 'XYZ Software solutions', 'effective_date': '1st April 2023', 'end_date': '31st March 2024', 'contract_value': 'USD 35000'}, name='save_contract_details', type='tool_use', toolset_name=None)


If Claude wants to use a tool, we’ll see a `tool_use` block.

```text
Claude did NOT save anything.

Claude requested:

"Please call save_contract_details
with these arguments."
```

Our Python program still has to execute the actual function.

`tool_use` is a request to take action — not the action itself.


#### Extract the Tool Request

In [30]:
tool_use = next(
    block
    for block in response.content
    if block.type == "tool_use"
)

print("Tool name:")
print(tool_use.name)

print("\nArguments:")
print(tool_use.input)

Tool name:
save_contract_details

Arguments:
{'customer_name': 'XYZ Software solutions', 'effective_date': '1st April 2023', 'end_date': '31st March 2024', 'contract_value': 'USD 35000'}


We extract the tool_use block from Claude’s response so our Python code can identify the requested tool and read the arguments Claude generated for it

Here’s another reliability issue.

Our Python function expects fields like:

```text
customer_name
effective_date
end_date
contract_value
```

But Claude might generate:

```json
{
  "customer": "XYZ Software Solutions",
  "value": 35000
}
```

A human understands this.

Our function does not.


for that we only need to add:

```python
"strict": True
```

to the tool definition.

> **Result:** Claude must follow the tool’s defined input schema more precisely.


In [31]:
tools = [
    {
        "name": "save_contract_details",
        "description": "Save key information extracted from a contract.",

        "strict": True,

        "input_schema": {
            "type": "object",
            "properties": {
                "customer_name": {
                    "type": "string"
                },
                "effective_date": {
                    "type": "string"
                },
                "end_date": {
                    "type": "string"
                },
                "contract_value": {
                    "type": "string"
                }
            },
            "required": [
                "customer_name",
                "effective_date",
                "end_date",
                "contract_value"
            ],
            "additionalProperties": False
        }
    }
]

In [33]:
response = client.messages.create(
    model=MODEL,
    max_tokens=800,
    system="""
Extract key contract information from the contract.

Use only information explicitly present in the contract.

Save the extracted information using the available tool.
""",
    messages=[
        {
            "role": "user",
            "content": f"""
<contract>
{document}
</contract>
"""
        }
    ],
    tools=tools
)

In [34]:
tool_use = next(
    block
    for block in response.content
    if block.type == "tool_use"
)

print(tool_use.input)

{'customer_name': 'XYZ Software solutions', 'effective_date': '1st April 2023', 'end_date': '31st March 2024', 'contract_value': 'USD 35000'}


## Complete the Tool Loop

This completes the tool-use loop by telling Claude what happened after its requested tool was actually executed.

Claude has already extracted the contract details and requested the `save_contract_details` tool.

Our Python function executed the save successfully.

But Claude does not automatically know that the tool succeeded.

So we need to send the tool result back to Claude.

```text
Contract
   ↓
Claude extracts details
   ↓
Claude requests tool
   ↓
Python executes tool
   ↓
Send result back to Claude

In [38]:
messages = [
    {
        "role": "user",
        "content": f"""
<contract>
{document}
</contract>
"""
    },
    {
        "role": "assistant",
        "content": response.content
    },
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": json.dumps(tool_use.input)
            }
        ]
    }
]

In [39]:
final_response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    messages=messages,
    tools=tools
)

for block in final_response.content:
    if block.type == "text":
        print(block.text)

Perfect! I've successfully extracted and saved the key contract details from the AWS Customer Agreement:

**Contract Summary:**
- **Customer Name:** XYZ Software solutions
- **Effective Date:** 1st April 2023
- **End Date:** 31st March 2024
- **Contract Value:** USD 35,000 (paid annually before the start of work)
- **Contract Duration:** 12 months
- **Auto-renewal:** No, this contract does not renew automatically after it reaches the end date

The contract is a standard AWS Customer Agreement between Amazon Web Services and XYZ Software solutions for the provision of cloud computing services.


## What If the Tool Fails?

So far, the tool worked successfully.

But **strict tool use only ensures that Claude sends valid arguments**.

It cannot guarantee that the real database, API, or external service is working.

For example:

```text
Correct tool arguments ✅
        ↓
Database unavailable ❌

In [46]:
def save_contract_details(
    customer_name,
    effective_date,
    end_date,
    contract_value
):
    raise Exception("Database connection unavailable")

Now let's try to execute the same tool again.

This time, instead of letting the notebook crash, we'll catch the error.

In [47]:
try:
    tool_result = save_contract_details(
        **tool_use.input
    )

    is_error = False

except Exception as e:
    tool_result = {
        "status": "error",
        "message": str(e)
    }

    is_error = True

print(tool_result)

{'status': 'error', 'message': 'Database connection unavailable'}


### Expected Output

```text
{
    'status': 'error',
    'message': 'Database connection unavailable'
}

In [52]:
error_messages = [
    {
        "role": "user",
        "content": f"""
<contract>
{document}
</contract>
"""
    },
    {
        "role": "assistant",
        "content": response.content
    },
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": json.dumps(tool_result),
                "is_error": True
            }
        ]
    }
]

In [53]:
error_response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    messages=error_messages,
    tools=tools
)

for block in error_response.content:
    if block.type == "text":
        print(block.text)

I encountered a database connection error, but I was able to extract the following key contract details from the AWS Customer Agreement:

**Contract Summary:**
- **Customer Name:** XYZ Software solutions
- **Effective Date:** 1st April 2023
- **End Date:** 31st March 2024
- **Contract Value:** USD 35,000 (paid annually before the start of work)
- **Duration:** 12 months
- **Auto-Renewal:** Does not renew automatically after the end date

**Additional Key Terms:**
- Governing Parties: Amazon Web Services and XYZ Software solutions
- Last Updated: April 20, 2023
- Payment Terms: Annual payment of USD 35,000 before work commencement
- Currency: USD 35,000 (Note: For India-based customers, this is converted to INR equivalent)
- Termination: Either party can terminate for convenience with appropriate notice periods
- No automatic renewal after March 31, 2024

The database is currently unavailable, so the information could not be persisted. Please try again when the database connection is re

## What Happened?

Claude successfully understood and extracted the contract information.

The tool call itself was also correctly formed.

However, the actual Python function failed because we intentionally simulated a database connection error.

Because we sent that failure back using `is_error: True`, Claude now knows that the save did **not** succeed.

So the workflow became:

```text
Contract
   ↓
Claude extracts details ✅
   ↓
Claude creates valid tool call ✅
   ↓
Python tries to save
   ↓
Database fails ❌
   ↓
Error is returned to Claude
   ↓
Claude explains that the data was extracted,
but could not be saved